# PINNs Journey - Quickstart

This notebook demonstrates the basic workflow for training a PINN using this library.

In [ ]:
# Install if needed
# !pip install -e .

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

## 1. Define the Problem

Let's solve the 1D Heat Equation:

$$u_t = \alpha u_{xx}, \quad x \in [0,1], t \in [0,1]$$

$$u(0,t) = u(1,t) = 0, \quad u(x,0) = \sin(\pi x)$$

Analytical: $$u(x,t) = e^{-\alpha\pi^2 t} \sin(\pi x)$$

In [ ]:
from src.pinns.equations import get_equation
from src.pinns.models import MLP
from src.pinns.losses import create_pinn_loss
from src.pinns.training import Trainer, TrainingConfig
from src.pinns.evaluation import evaluate_model, plot_solution_1d, plot_loss_history

# Create equation
equation = get_equation('heat_1d', alpha=0.01)

# Create model
model = MLP(input_dim=2, output_dim=1, hidden_dims=[64, 64, 64, 64], activation='tanh')

# Create loss function
loss_fn = create_pinn_loss(equation)

# Training config
config = TrainingConfig(
    epochs=5000,
    learning_rate=1e-3,
    optimizer='adam',
    scheduler='cosine',
    log_freq=500,
    save_freq=1000,
    early_stopping_patience=1000,
)

# Create trainer
trainer = Trainer(model, equation, loss_fn, config)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Train the Model

In [ ]:
# Train
history = trainer.train()

## 3. Evaluate and Visualize

In [ ]:
# Evaluate on test grid
results = evaluate_model(model, equation, analytical_solution=equation.analytical_solution)

print(f"Relative L2 Error: {results['metrics']['l2_relative']:.4e}")
print(f"Max Error: {results['metrics']['linf']:.4e}")

In [ ]:
# Plot results
fig = plot_solution_1d(results, equation, show=True)
fig = plot_loss_history(history, show=True)

## 4. Try Other Equations

In [ ]:
# List available equations
from src.pinns.equations import list_equations
print("Available equations:", list_equations())

## 5. Next Steps

- Try `notebooks/01_autograd_fundamentals.ipynb` for autograd deep dive
- Try `notebooks/02_first_pinn_ode.ipynb` for first PINN from scratch
- Try `notebooks/03_heat_equation.ipynb` for detailed heat equation
- Try `notebooks/04_burgers_equation.ipynb` for nonlinear PDE
- Read `docs/01_sci_ml_overview.md` for SciML overview
- Read `docs/02_math_foundations.md` for PDE theory